In [9]:
import numpy as np        

In [10]:
# ----------------------------
# Utility: Stable Softmax
# ----------------------------
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(x)
    return exp / np.sum(exp, axis=-1, keepdims=True)

In [ ]:
# ----------------------------
# Scaled Dot Product Attention
# ----------------------------
def scaled_dot_product(q, k, v, use_mask=False):
    """
    q, k, v shape:
    (batch_size, num_heads, seq_len, head_dim)
    """

    """
    step1: to find the scaled dot product attention, we first compute the dot product 
    between the query and key which is known as the scores.

    step2: we then normalize the scores by the square root of the key dimension.

    step3: we then apply the softmax function to the scores to get the attention weights.

    step4: we then multiply the attention weights with the value to get the final output.

    optional step if we are in decoder: we then apply a mask to the scores to avoid attention to future tokens.
    """
    # d_k = q.shape[-1]
    d_k = self.d_model // self.num_heads
    
    scores = np.matmul(q, np.transpose(k, (0,1,3,2))) / np.sqrt(d_k)
    # (batch, heads, seq_len, seq_len)
        # 🔹 Optional Masking
    if use_mask:  ## Onli for decoder
        batch_size, num_heads, seq_len, _ = scores.shape
        
        # Create causal mask
        mask = np.triu(
            np.ones((seq_len, seq_len)) * -1e9,
            k=1
        )
        # Expand mask for batch and heads
        mask = mask.reshape(1, 1, seq_len, seq_len)
        scores += mask

    attention = softmax(scores)
    values = np.matmul(attention, v)

    return values, attention

In [ ]:
# ----------------------------
# Multi-Head Attention Class
# ----------------------------
class MultiHeadAttention:
    
    def __init__(self, input_dim=512, d_model=512, num_heads=8):

        """
        we are having multiple words and for each word there are 512 dimensions 

        and we are having 8 heads

        and then model dimension as 512 andthat can be anything like 256 , 512 ,1024 etc

        as input_dim=512, d_model=512, num_heads=8 then each head will have 64 dimensions(inp dim % num_heads)
        
        Then we take the W_qkv and W_o , W_qkv is 512x1536 and W_o is 512x512

        W_qkv is multiplied with x then it forms q,k,v(in the W_qkv matrix there are 3 matrices q,k,v together 
        that is why the shape is 512x1536)

        Later when the 8 attention heads are produced for each word they are concatinated together 
        and then it is multiplied with W_o to get the final output. 
        """
        
        assert d_model % num_heads == 0
        
        self.input_dim = input_dim
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.W_qkv = np.random.randn(input_dim, 3 * d_model)
        self.W_o = np.random.randn(d_model, d_model)
    
    
    def forward(self, x, use_mask=False):
        """
        x shape:
        (batch_size, seq_len, input_dim)
        """
        
        batch_size, seq_len, _ = x.shape
        
        # 1️⃣ Linear projection
        qkv = np.matmul(x, self.W_qkv)
        
        
        # 2️⃣ Reshape for multi-head
        qkv = qkv.reshape(batch_size,
                          seq_len,
                          self.num_heads,
                          3 * self.head_dim)
        
        
        # Move heads forward
        qkv = np.transpose(qkv, (0, 2, 1, 3))

        # Split Q, K, V
        q, k, v = np.split(qkv, 3, axis=-1)

        # Apply attention
        values, attention = scaled_dot_product(q, k, v, use_mask)

        # Concatenate heads
        values = np.transpose(values, (0, 2, 1, 3))
        values = values.reshape(batch_size, seq_len, self.d_model)

        # This is Z (before final projection)
        Z = values

        # Final projection
        out = np.matmul(Z, self.W_o)
        print(qkv)

In [15]:
batch_size = 2
seq_len = 4
input_dim = 512

# Random sample input
x = np.random.randn(batch_size, seq_len, input_dim)

print("Input shape:", x.shape)

Input shape: (2, 4, 512)


In [16]:
# Create model
mha = MultiHeadAttention(
    input_dim=512,
    d_model=512,
    num_heads=8
)

# Forward pass without masking
out, attention = mha.forward(x, use_mask=False)

print("Output shape:", out.shape)
print("Attention shape:", attention.shape)

[[[[ -9.02259834  21.40406167  30.98591457 ...  -9.56880335
    -17.54696244  -1.32478618]
   [-21.52878698   5.14333188   2.378474   ...  -7.19493862
      9.40243467 -16.88692758]
   [-24.51921801   0.99326191  -5.24598813 ...  15.18498604
     -4.94100969   0.22049155]
   [-13.10983884 -28.83237998 -18.35158894 ...  32.586602
     28.13273547  -9.7583404 ]]

  [[-51.42289542 -59.58194412  20.37532773 ...  16.94770913
     61.48458967  48.1274817 ]
   [ 29.0527006   -6.446968   -20.93713883 ...  11.33799835
     -6.2310841   -9.71547926]
   [-27.36562001  -4.8697674  -29.87538916 ... -16.01794785
    -33.98003837 -34.53734247]
   [ -3.68405918  -9.6086051   -4.01977416 ... -11.44852971
     63.73718087 -16.21328915]]

  [[-51.66646339  15.04638799   9.9745085  ...  -3.01243348
     -6.46304469 -14.66852365]
   [ 20.26011284  16.64035433  33.51547247 ...  47.95188407
    -17.47391096 -15.37580567]
   [-14.00068093  11.18427149  -1.66353269 ...  30.09084366
    -14.95195135   3.2905527

TypeError: cannot unpack non-iterable NoneType object